In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from models import SimpleNet
import torch
import torch.nn as nn
import mlflow
import numpy as np
from torchmetrics.classification import BinaryAUROC


In [40]:
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print(f"Using CPU")
    device = torch.device("cpu")


Using GPU: NVIDIA GeForce GTX 1650


In [41]:
!nvidia-smi


Sun Sep 13 18:50:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.83                 Driver Version: 572.83         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P0             14W /   35W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [42]:
df = pd.read_csv("data/train-cat-encoded.csv")
print("Training set size: ", df.shape)
y = df['Will_Buy_EV']
x = df.drop(columns=['Will_Buy_EV', 'id'])
print(x.shape)


Training set size:  (668665, 22)
(668665, 20)


### Continous feature transformation
TODO
- Log transform
- standardisation
- min-max normalisation
- quantile binning (Daily_Commute_km)
- clipping/winsorizing

In [43]:
transformed_x = x.copy()
# min_max_col = []
log_transform = ['Annual_Income_USD']
standard_transform = [
    'Age', 'Environmental_Concern_Level', 
    'Number_of_Cars_Owned', 'Range_Anxiety_Level', 
    'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 
    'Daily_Commute_km'
    ]

# for col in min_max_col:
#     transformed_x[f"{col}_min_max"] = (transformed_x[col] - np.min(transformed_x[col])) / (np.max(transformed_x[col] - np.min(transformed_x[col])))
#     transformed_x = transformed_x.drop(columns=col)

for col in log_transform:
    transformed_x[f"{col}_log_trans"] = np.log1p(transformed_x[col])
    transformed_x = transformed_x.drop(columns=col)

for col in standard_transform:
    transformed_x[f"{col}_std_trans"] = (transformed_x[col] - np.mean(transformed_x[col])) / np.std(transformed_x[col])
    transformed_x = transformed_x.drop(columns=col)


In [44]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
subsetNum = 50000
subset_x = x.iloc[:subsetNum]
subset_y = y.iloc[:subsetNum]


In [45]:
x_tensor = torch.as_tensor(subset_x.to_numpy(), dtype=torch.float16)
y_tensor = torch.as_tensor(subset_y.to_numpy(), dtype=torch.float16)
n_sample, n_feature = x_tensor.shape
print(n_sample, n_feature)


50000 20


In [ ]:
# Training Parameters
params = {
    'lr': 0.01,
    'epoch': 5,
    'hidden_nodes': 10,
    'batch_size': 32,
    'training_sample': n_sample,
    'training_features': n_feature
}


In [ ]:
model = SimpleNet(params['training_features'], 10, 2).to(device)
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(model.parameters(), lr=params['lr'])
auc = BinaryAUROC()

for name, param in model.named_parameters():
    print(f"{name}: {param.dtype}")


layer1.weight: torch.float16
layer2.weight: torch.float16


In [ ]:
mlflow.set_experiment("Neural Network Experiment")
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)

bs = params['batch_size']

with mlflow.start_run() as run:
    mlflow.log_params(params)

    for epoch in range(params['epoch']):
        total_loss = 0
        
        for batch_idx in range(0, params['n_sample'], bs):
            feature_mat = x_tensor[batch_idx:batch_idx+bs].to(device)
            true_label = y_tensor[batch_idx:batch_idx+bs].to(device)

            #Forward pass
            optimiser.zero_grad()
            pred_label = model.forward(feature_mat)

            #Calculate metrics
            loss = loss_fn(pred_label, true_label)
            auc_score = auc(pred_label, true_label)
            total_loss += loss

            #Propagate backwards
            loss.backwards()
            optimiser.step()

            # if batch_idx % 50 == 0:
            batch_loss = total_loss / (batch_idx + 1)
            mlflow.log_metrics(
                {
                    'loss': loss,
                    'auc_roc': auc_score,
                    "batch_loss": batch_loss
                    }
            )

        #Log checkpoint at the end of each epoch
        mlflow.pytorch.log_model(model, name=f'checkpoint_{epoch}')

    model_info = mlflow.pytorch.log_model(model, name="final_model")  



            